In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = "../data/fake_job_postings.csv"

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
df.info()

In [ ]:
df["fraudulent"].value_counts()

In [ ]:
df["fraudulent"].value_counts(normalize=True) * 100

In [ ]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

df[text_columns].isnull().sum()

In [ ]:
df["text"] = df[text_columns].fillna("").agg(" ".join, axis=1)

df[["text", "fraudulent"]].head()

In [ ]:
X = df["text"]
y = df["fraudulent"]

print("Number of samples:", len(X))
print("Number of real jobs:", (y == 0).sum())
print("Number of fake jobs:", (y == 1).sum())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        stop_words="english",
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

y_pred = baseline_model.predict(X_test)
y_proba = baseline_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("Baseline Model Results")
print("----------------------")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Real Job", "Fake Job"]))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Real Job", "Fake Job"],
    cmap="Blues"
)

plt.title("Baseline Model Confusion Matrix")
plt.show()

In [ ]:
example_job = """
We are hiring remote workers immediately. No experience needed.
Earn $5000 per week from home. Training provided.
Send your personal details and payment information to get started.
"""

prediction = baseline_model.predict([example_job])[0]
probability = baseline_model.predict_proba([example_job])[0][1]

print("Prediction:", "Fake/Suspicious" if prediction == 1 else "Real")
print("Scam probability:", round(probability, 4))

In [ ]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(baseline_model, "../models/baseline_model.pkl")

print("Baseline model saved successfully.")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np

X_train_text, X_val_text, y_train_final, y_val = train_test_split(
    X_train.astype(str),
    y_train.astype("float32"),
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

X_train_tf = tf.convert_to_tensor(X_train_text.tolist(), dtype=tf.string)
X_val_tf = tf.convert_to_tensor(X_val_text.tolist(), dtype=tf.string)
X_test_tf = tf.convert_to_tensor(X_test.astype(str).tolist(), dtype=tf.string)

y_train_final_np = y_train_final.to_numpy(dtype="float32")
y_val_np = y_val.to_numpy(dtype="float32")
y_test_np = y_test.to_numpy(dtype="float32")

print(X_train_tf.shape, X_train_tf.dtype)
print(y_train_final_np.shape, y_train_final_np.dtype)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

max_tokens = 20000
sequence_length = 300

vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length,
    standardize="lower_and_strip_punctuation"
)

vectorizer.adapt(X_train_tf)

In [ ]:
inputs = keras.Input(shape=(), dtype=tf.string)

x = vectorizer(inputs)

x = layers.Embedding(
    input_dim=max_tokens,
    output_dim=128,
    mask_zero=True
)(x)

x = layers.Bidirectional(layers.LSTM(64))(x)

x = layers.Dropout(0.4)(x)

x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

keras_model = keras.Model(inputs, outputs)

keras_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

keras_model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train_final_np.astype(int)
)

class_weights = {
    0: class_weights_array[0],
    1: class_weights_array[1]
}

class_weights

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = keras_model.fit(
    X_train_tf,
    y_train_final_np,
    validation_data=(X_val_tf, y_val_np),
    epochs=8,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

In [ ]:
keras_probs = keras_model.predict(X_test_tf).ravel()
keras_preds = (keras_probs >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

keras_accuracy = accuracy_score(y_test_np, keras_preds)
keras_precision = precision_score(y_test_np, keras_preds)
keras_recall = recall_score(y_test_np, keras_preds)
keras_f1 = f1_score(y_test_np, keras_preds)
keras_roc_auc = roc_auc_score(y_test_np, keras_probs)

print("Keras Model Results")
print("-------------------")
print("Accuracy:", round(keras_accuracy, 4))
print("Precision:", round(keras_precision, 4))
print("Recall:", round(keras_recall, 4))
print("F1-score:", round(keras_f1, 4))
print("ROC-AUC:", round(keras_roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test_np, keras_preds, target_names=["Real Job", "Fake Job"]))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test_np,
    keras_preds,
    display_labels=["Real Job", "Fake Job"],
    cmap="Blues"
)

plt.title("Keras Model Confusion Matrix")
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Model": ["TF-IDF + Logistic Regression", "Keras BiLSTM"],
    "Accuracy": [accuracy, keras_accuracy],
    "Precision": [precision, keras_precision],
    "Recall": [recall, keras_recall],
    "F1-score": [f1, keras_f1],
    "ROC-AUC": [roc_auc, keras_roc_auc]
})

comparison

In [ ]:
example_job = """
We are hiring remote workers immediately. No experience needed.
Earn $5000 per week from home. Training provided.
Send your personal details and payment information to get started.
"""

example_tf = tf.convert_to_tensor([example_job], dtype=tf.string)

probability = keras_model.predict(example_tf)[0][0]

print("Scam probability:", round(float(probability), 4))

if probability >= 0.5:
    print("Prediction: Fake/Suspicious Job")
else:
    print("Prediction: Real Job")

In [ ]:
import os
import json
import numpy as np

os.makedirs("../models", exist_ok=True)

# 1. Save model weights
keras_model.save_weights("../models/keras_job_scam_model.weights.h5")

# 2. Save vectorizer vocabulary using UTF-8
vocab = vectorizer.get_vocabulary()

with open("../models/vectorizer_vocabulary.txt", "w", encoding="utf-8") as f:
    for word in vocab:
        f.write(word + "\n")

# 3. Save model config values
config = {
    "max_tokens": max_tokens,
    "sequence_length": sequence_length,
    "embedding_dim": 128,
    "lstm_units": 64
}

with open("../models/model_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print("Model weights, vectorizer vocabulary, and config saved successfully.")

In [ ]:
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Load config
with open("../models/model_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

# Load vocabulary
with open("../models/vectorizer_vocabulary.txt", "r", encoding="utf-8") as f:
    loaded_vocab = [line.rstrip("\n") for line in f]

# Recreate vectorizer
loaded_vectorizer = layers.TextVectorization(
    max_tokens=config["max_tokens"],
    output_mode="int",
    output_sequence_length=config["sequence_length"],
    standardize="lower_and_strip_punctuation",
    vocabulary=loaded_vocab
)

# Recreate model architecture
inputs = keras.Input(shape=(), dtype=tf.string)

x = loaded_vectorizer(inputs)

x = layers.Embedding(
    input_dim=config["max_tokens"],
    output_dim=config["embedding_dim"],
    mask_zero=True
)(x)

x = layers.Bidirectional(layers.LSTM(config["lstm_units"]))(x)

x = layers.Dropout(0.4)(x)

x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

loaded_model = keras.Model(inputs, outputs)

loaded_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc")
    ]
)

# Build model once before loading weights
dummy_input = tf.convert_to_tensor(["test job post"], dtype=tf.string)
loaded_model(dummy_input)

# Load weights
loaded_model.load_weights("../models/keras_job_scam_model.weights.h5")

print("Model loaded successfully.")

In [ ]:
test_job = """
Remote data entry assistant needed. Flexible hours.
No degree required. Weekly payment. Apply now.
"""

test_tf = tf.convert_to_tensor([test_job], dtype=tf.string)

loaded_probability = loaded_model.predict(test_tf)[0][0]

print("Loaded model scam probability:", round(float(loaded_probability), 4))

if loaded_probability >= 0.5:
    print("Prediction: Fake/Suspicious Job")
else:
    print("Prediction: Real Job")

In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd

precisions, recalls, thresholds = precision_recall_curve(y_test_np, keras_probs)

# thresholds has one fewer value than precisions/recalls
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)

best_f1_index = np.argmax(f1_scores)
best_f1_threshold = thresholds[best_f1_index]

print("Best F1 threshold:", round(float(best_f1_threshold), 4))
print("Precision at best F1:", round(float(precisions[best_f1_index]), 4))
print("Recall at best F1:", round(float(recalls[best_f1_index]), 4))
print("Best F1-score:", round(float(f1_scores[best_f1_index]), 4))

In [ ]:
target_recall = 0.90

valid_indices = np.where(recalls[:-1] >= target_recall)[0]

if len(valid_indices) > 0:
    best_recall_index = valid_indices[np.argmax(precisions[valid_indices])]
    recall_priority_threshold = thresholds[best_recall_index]

    print("Recall-priority threshold:", round(float(recall_priority_threshold), 4))
    print("Precision:", round(float(precisions[best_recall_index]), 4))
    print("Recall:", round(float(recalls[best_recall_index]), 4))
    print("F1-score:", round(float(f1_scores[best_recall_index]), 4))
else:
    print("No threshold found with recall >=", target_recall)

In [ ]:
chosen_threshold = best_f1_threshold

tuned_preds = (keras_probs >= chosen_threshold).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

tuned_accuracy = accuracy_score(y_test_np, tuned_preds)
tuned_precision = precision_score(y_test_np, tuned_preds)
tuned_recall = recall_score(y_test_np, tuned_preds)
tuned_f1 = f1_score(y_test_np, tuned_preds)
tuned_roc_auc = roc_auc_score(y_test_np, keras_probs)

print("Tuned Keras Model Results")
print("-------------------------")
print("Threshold:", round(float(chosen_threshold), 4))
print("Accuracy:", round(tuned_accuracy, 4))
print("Precision:", round(tuned_precision, 4))
print("Recall:", round(tuned_recall, 4))
print("F1-score:", round(tuned_f1, 4))
print("ROC-AUC:", round(tuned_roc_auc, 4))

print("\nClassification Report:")
print(classification_report(y_test_np, tuned_preds, target_names=["Real Job", "Fake Job"]))

In [ ]:
threshold_comparison = pd.DataFrame({
    "Model": ["Keras Default Threshold", "Keras Tuned Threshold"],
    "Threshold": [0.5, chosen_threshold],
    "Accuracy": [keras_accuracy, tuned_accuracy],
    "Precision": [keras_precision, tuned_precision],
    "Recall": [keras_recall, tuned_recall],
    "F1-score": [keras_f1, tuned_f1],
    "ROC-AUC": [keras_roc_auc, tuned_roc_auc]
})

threshold_comparison

In [ ]:
import json

threshold_config = {
    "default_threshold": 0.5,
    "high_risk_threshold": float(chosen_threshold),
    "threshold_strategy": "default threshold for suspicious, best F1 threshold for high-risk classification"
}

with open("../models/threshold_config.json", "w", encoding="utf-8") as f:
    json.dump(threshold_config, f, indent=4)

print("Threshold config saved successfully.")

In [ ]:
def predict_job_risk(job_text, model, low_threshold=0.5, high_threshold=0.9345):
    job_tf = tf.convert_to_tensor([job_text], dtype=tf.string)
    probability = float(model.predict(job_tf, verbose=0)[0][0])

    if probability >= high_threshold:
        risk_level = "High Risk"
        prediction = "Likely Scam"
    elif probability >= low_threshold:
        risk_level = "Medium Risk"
        prediction = "Suspicious"
    else:
        risk_level = "Low Risk"
        prediction = "Likely Legitimate"

    return {
        "scam_probability": round(probability, 4),
        "risk_level": risk_level,
        "prediction": prediction
    }

In [ ]:
test_job_1 = """
We are hiring remote workers immediately. No experience needed.
Earn $5000 per week from home. Training provided.
Send your personal details and payment information to get started.
"""

predict_job_risk(test_job_1, keras_model)

In [ ]:
test_job_2 = """
Software Engineering Intern needed for a technology company.
Responsibilities include building backend APIs, writing unit tests,
working with Git, and collaborating with senior engineers.
Applicants should know Python, JavaScript, and basic databases.
"""

predict_job_risk(test_job_2, keras_model)

In [ ]:
import re

def detect_warning_signs(job_text):
    text = job_text.lower()
    warning_signs = []

    patterns = {
        "Unrealistic salary or income promise": [
            r"\$\s?\d{4,}",
            r"\d{4,}\s?(usd|dollars|per week|weekly)",
            r"earn\s+\$?\d+",
            r"make\s+\$?\d+",
            r"high income",
            r"unlimited income"
        ],

        "Requests payment or financial information": [
            r"payment information",
            r"bank account",
            r"wire transfer",
            r"western union",
            r"registration fee",
            r"processing fee",
            r"upfront fee",
            r"send money",
            r"deposit"
        ],

        "Asks for sensitive personal information": [
            r"social security",
            r"passport",
            r"id card",
            r"personal details",
            r"driver.?s license",
            r"date of birth",
            r"national id"
        ],

        "Too easy / no experience required": [
            r"no experience",
            r"no skills required",
            r"no degree required",
            r"work from home immediately",
            r"start immediately",
            r"easy job",
            r"simple work"
        ],

        "Urgency or pressure": [
            r"apply now",
            r"limited spots",
            r"urgent hiring",
            r"immediate start",
            r"hiring immediately",
            r"act fast"
        ],

        "Vague job/company description": [
            r"various tasks",
            r"general duties",
            r"flexible role",
            r"online tasks",
            r"work at your own pace",
            r"be your own boss"
        ]
    }

    for category, regex_list in patterns.items():
        for pattern in regex_list:
            if re.search(pattern, text):
                warning_signs.append(category)
                break

    return warning_signs

In [ ]:
def predict_job_risk_with_explanation(
    job_text,
    model,
    low_threshold=0.5,
    high_threshold=0.9345
):
    job_tf = tf.convert_to_tensor([job_text], dtype=tf.string)
    probability = float(model.predict(job_tf, verbose=0)[0][0])

    if probability >= high_threshold:
        risk_level = "High Risk"
        prediction = "Likely Scam"
    elif probability >= low_threshold:
        risk_level = "Medium Risk"
        prediction = "Suspicious"
    else:
        risk_level = "Low Risk"
        prediction = "Likely Legitimate"

    warning_signs = detect_warning_signs(job_text)

    return {
        "prediction": prediction,
        "risk_level": risk_level,
        "scam_probability": round(probability, 4),
        "warning_signs": warning_signs
    }

In [ ]:
suspicious_job = """
We are hiring remote workers immediately. No experience needed.
Earn $5000 per week from home. Training provided.
Send your personal details and payment information to get started.
Apply now, limited spots available.
"""

predict_job_risk_with_explanation(suspicious_job, keras_model)

In [ ]:
normal_job = """
Software Engineering Intern needed for a technology company.
Responsibilities include building backend APIs, writing unit tests,
working with Git, and collaborating with senior engineers.
Applicants should know Python, JavaScript, and basic databases.
"""

predict_job_risk_with_explanation(normal_job, keras_model)

In [ ]:
warning_rules = {
    "Unrealistic salary or income promise": [
        "$ amount above normal job-posting patterns",
        "weekly high income claims",
        "earn/make money promises"
    ],
    "Requests payment or financial information": [
        "payment information",
        "bank account",
        "wire transfer",
        "registration fee",
        "processing fee",
        "upfront fee"
    ],
    "Asks for sensitive personal information": [
        "passport",
        "ID card",
        "personal details",
        "driver's license",
        "date of birth",
        "national ID"
    ],
    "Too easy / no experience required": [
        "no experience",
        "no skills required",
        "no degree required",
        "easy job"
    ],
    "Urgency or pressure": [
        "apply now",
        "limited spots",
        "urgent hiring",
        "immediate start"
    ],
    "Vague job/company description": [
        "various tasks",
        "general duties",
        "online tasks",
        "work at your own pace"
    ]
}

with open("../models/warning_rules.json", "w", encoding="utf-8") as f:
    json.dump(warning_rules, f, indent=4)

print("Warning rules saved successfully.")

In [ ]:
import sys
print(sys.executable)